# Data Cleaning with Pandas

In this notebook we'll go through a few basic data cleaning steps that should be performed on all new datasets where necessary.

We'll go through the process with both the `orders` and `orderlines` datasets. You can then practice these skills by cleaning the `products` dataset yourself

In [1]:
import pandas as pd

In [2]:
# orders.csv
url = "https://drive.google.com/file/d/1Vu0q91qZw6lqhIqbjoXYvYAQTmVHh6uZ/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders = pd.read_csv(path)

# orderlines.csv
url = "https://drive.google.com/file/d/1FYhN_2AzTBFuWcfHaRuKcuCE6CWXsWtG/view?usp=drive_link"
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines = pd.read_csv(path)

Before we begin, let's create a copy of the `orders` and `orderlines` DataFrames. This way we are sure any of our changes won't affect the original DataFrames

In [3]:
orders_df = orders.copy()

In [4]:
orderlines_df = orderlines.copy()

One of the best ways to begin data cleaning is by exploring using `.info()`. This will tell us:
* The shape of the DataFrame
* The names of the columns
* If there are any missing values
* The datatypes of the columns

By exploring the missing values and correcting any incorrect datatypes, we often come across inconsistencies in our data.

Beyond this, we should also have a **check for any duplicate rows**.

Let's first deal with the duplicates, as it's nice and easy, then we'll explore what `.info()` has to tell us.

## 1.&nbsp; Duplicates
We can check for duplicates using the pandas [.duplicated()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) method.

We can then delete these rows, if we wish, using [.drop_duplicates()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)

In [5]:
# orders_df
orders_df.duplicated().sum()

np.int64(0)

In [6]:
# orderlines_df
orderlines_df.duplicated().sum()

np.int64(0)

We have no duplicate rows in either DataFrame. Easy, there is no problem to solve. Normally though, if there were some duplicates, we'd drop the extra rows.

# 2.&nbsp; `.info()`

In [7]:
orders_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  str    
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  str    
dtypes: float64(1), int64(1), str(2)
memory usage: 6.9 MB


* `total_paid` has 5 missing values
* `created_date` should become datetime datatype

In [8]:
orderlines_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                293983 non-null  int64
 1   id_order          293983 non-null  int64
 2   product_id        293983 non-null  int64
 3   product_quantity  293983 non-null  int64
 4   sku               293983 non-null  str  
 5   unit_price        293983 non-null  str  
 6   date              293983 non-null  str  
dtypes: int64(4), str(3)
memory usage: 15.7 MB


* `date` should be a datetime datatype
* `unit_price` should be a float datatype

## 3.&nbsp; Missing values

### 3.1.&nbsp; Orders
* `total_paid` has 5 missing values

In [9]:
num_missing = orders_df['total_paid'].isna().sum()
total_rows = orders_df.shape[0]
percent_missing = (100*num_missing/total_rows)
print(f"5 missing values represents {percent_missing:.4f}% of the rows in our DataFrame")

5 missing values represents 0.0022% of the rows in our DataFrame


> A quick way to find out a percentage if you don't need to print out a sentence for yourself/students/colleagues is `.value_counts(normalize=True)`

In [10]:
orders_df['total_paid'].isna().value_counts(normalize=True)

total_paid
False    0.999978
True     0.000022
Name: proportion, dtype: float64

As there is such a tiny amount of missing values, we will simply delete these rows, as we have enough data without them.

In [11]:
orders_df = orders_df.dropna(axis=0)

Should you have a significant number of missing values in the future, you have a choice:
+ you can impute the values
+ you can take the values from other DataFrames if they are redundantly stored
+ you can delete the rows or columns
+ or any number of other creative solutions

Please, always consider how much time you have on your project, and what impact your method of choice will have on your final assesment.

### 3.2.&nbsp; Orderlines
There are no missing values in `orderlines_df`

## 4.&nbsp; Datatypes

### 4.1.&nbsp; Orders
* `created_date` should become datetime datatype

In [12]:
orders_df.created_date = pd.to_datetime(orders_df.created_date)

In [13]:
orders_df["created_date"] = pd.to_datetime(orders_df["created_date"])

### 4.1.&nbsp; Orderlines
* `date` should be a datetime datatype
* `unit_price` should be a float datatype

#### 4.1.1.&nbsp; `date`

In [14]:

orderlines_df = orderlines.copy()
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])

#### 4.1.2.&nbsp;`unit_price`

In [15]:
orderlines_df.id.nunique()

293983

In [16]:

orderlines_df = orderlines.copy()
orderlines_df["date"] = pd.to_datetime(orderlines_df["date"])
exp_df = orderlines_df.copy()
exp_df['unit_price'] = pd.to_numeric(exp_df['unit_price'], errors='coerce')
id_nan = exp_df.loc[exp_df['unit_price'].isna(), 'id']
nan_price =  orderlines_df.copy()
nan_price = orderlines_df[orderlines_df['id'].isin(id_nan)].copy()
nan_price


,id,id_order,product_id,product_quantity,sku,unit_price,date
6,1119115,299544,0,1,APP1582,1.137.99,2017-01-01 01:17:21
11,1119126,299549,0,1,PAC0929,2.565.99,2017-01-01 02:07:42
15,1119131,299553,0,1,APP1854,3.278.99,2017-01-01 02:14:47
43,1119195,299582,0,1,PAC0961,2.616.99,2017-01-01 08:54:00
59,1119214,299596,0,1,PAC1599,2.873.99,2017-01-01 09:53:11
...,...,...,...,...,...,...,...
293862,1649999,452946,0,1,APP2075,2.999.00,2018-03-14 13:03:33
293887,1650045,527321,0,1,PAC2148,3.497.00,2018-03-14 13:10:15
293889,1650050,527324,0,1,PAC2117,3.075.00,2018-03-14 13:10:56
293911,1650088,527342,0,1,APP2492,1.329.00,2018-03-14 13:24:51


In [17]:
def fix_unit_price(price):
    length = len(price)
    if length == 8:
        return price[:1] + price[2:]
    elif length == 9:
        return price[:2] + price[3:]
    elif length == 10:
        return price[:3] + price[4:]
    return price  # leave unchanged if no rule matches

nan_price['unit_price'] = nan_price['unit_price'].apply(fix_unit_price)
nan_price['unit_price'] = pd.to_numeric(nan_price['unit_price'], errors='coerce')

In [18]:

#nan_price['unit_price'] = nan_price['unit_price'].str[0] + nan_price['unit_price'].str[2:]
nan_price['unit_price'] = pd.to_numeric(nan_price['unit_price'])
nan_price.unit_price.isna().sum()

np.int64(0)

In [19]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"], errors='coerce')
orderlines_df.update(nan_price)
orderlines_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   id                293983 non-null  int64         
 1   id_order          293983 non-null  int64         
 2   product_id        293983 non-null  int64         
 3   product_quantity  293983 non-null  int64         
 4   sku               293983 non-null  str           
 5   unit_price        293983 non-null  float64       
 6   date              293983 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(4), str(1)
memory usage: 15.7 MB


As you can see when we try to convert `unit_price` to a numerical datatype, we receive a `ValueError` telling us that pandas doesn't understand the number `1.137.99`. This is probably because numbers cannot have multiple decimal points. Let's see if there are any other numbers like this:

> `.` is a wildcard in regex, we need the `\` as an escape

We skip the next 3 lines of code and go directly to the check for NaN below.

In [20]:
# Count the number of decimal points in the unit_price
#orderlines_df['unit_price'].str.count(r"\.").value_counts()

Looks like over 36000 rows in `orderlines` are affected by this problem. Let's work out how much that is as a percentage of our total data.

In [25]:
# # Count the rows with more than one `.`
# mult_decimal_rows = (orderlines_df['unit_price'].str.count(r"\.")>1).sum()

# # Find the percentage of corrupted rows
# percent_corrupted = (100 * mult_decimal_rows / orderlines_df.shape[0])
# print(f"{percent_corrupted:.2f}% of the rows in our DataFrame have multiple decimal points in the unit_price")

This is a bit of a tricky decision as 12.3% is a significant amount of our data... and we might even end up losing a larger portion of our data than this too. For the moment we will delete the rows as we only have 2 weeks for this project and I'd like some quick, accurate results to show. If we have time at the end, we can come back and investigate this problem further, maybe there's a solution?

Each row of `orderlines` represents a product in an order. For example, if order number 175 contained 3 seperate products, then order 175 would have 3 rows in `orderlines`, one row for each of the products. If 2 of those products have 'normal' prices (14.99, 15.85) and 1 has a price with 2 decimal points (1.137.99), we need to remove the whole order and not just the affected row. If we only remove the row with 2 decimal places then any later analysis about products and prices could be misleading.

We therefore need to find the order numbers associated with the rows that have 2 decimal points, and then remove all the associated rows.

In [ ]:
# # Boolean mask to find the orders that contain a price with multiple decimal points
# multiple_decimal_mask = orderlines_df['unit_price'].str.count(r"\.") > 1

# # Apply the boolean mask to the orderlines DataFrame. This way we can find the order_id of all the affected orders.
# corrupted_order_ids = orderlines_df.loc[multiple_decimal_mask, "id_order"]

# # Keep only the rows that do not have multiple decimal points
# orderlines_df = orderlines_df.loc[~orderlines_df['id_order'].isin(corrupted_order_ids)]

In [23]:
orderlines_df.unit_price.sort_values()
orderlines_df.unit_price.isna().sum()
# if we want we save the df back to csv in the next line
#orderlines_df.to_csv('../data/orderlines_full.csv', index=False)

np.int64(0)

We still have 216250 rows in orderlines to work with. This should be more than enough for our evaluation.

Now that all of the 2 decimal point prices have been removed, let's try again to convert the column `unit_price` to the correct datatype.

In [26]:
orderlines_df["unit_price"] = pd.to_numeric(orderlines_df["unit_price"])

It worked perfectly

# Challenge: Clean the `products` DataFrame
Now it's your turn. Use the lessons you learnt above and clean the products DataFrame. You don't have to copy exactly what we did. Think about the consequences of your actions, sometimes it is ok to delete rows, other times you may wish to come up with more creative solutions.

In [ ]:
# products.csv
#url = "https://drive.google.com/file/d/1afxwDXfl-7cQ_qLwyDitfcCx3u7WMvkU/view?usp=drive_link"
#path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
#as we work offline we use the data from the local folder
products = pd.read_csv('../data/products.csv')

In [37]:
products_df=products.copy()
products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   sku          19326 non-null  str  
 1   name         19326 non-null  str  
 2   desc         19319 non-null  str  
 3   price        19280 non-null  str  
 4   promo_price  19326 non-null  str  
 5   in_stock     19326 non-null  int64
 6   type         19276 non-null  str  
dtypes: int64(1), str(6)
memory usage: 1.0 MB


### Look for Duplicates

In [43]:
#check
products_df.duplicated().sum() #8746
# drop
products_df = products_df.drop_duplicates()
# re-check
products_df.duplicated().sum() #0

np.int64(0)

### Look for Missing values


In [46]:
#check
products_df.isna().sum()
#   price          46
#   type           50
# convert to numeric WITH errors as NaN
products_df.price = pd.to_numeric(products_df.price, errors='coerce')
products_df.promo_price = pd.to_numeric(products_df.promo_price, errors='coerce')
#check again
products_df.isna().sum()

sku               0
name              0
desc              7
price           423
promo_price    4597
in_stock          0
type             50
dtype: int64

In [49]:
# doesnt look good, we drop the promo_price completly
products_df.drop(columns='promo_price', inplace=True)
# and we drop the lines with NaN 
products_df = products_df.dropna(axis=0)
products_df.isna().sum()

sku         0
name        0
desc        0
price       0
in_stock    0
type        0
dtype: int64

### Check / Change Data types

In [50]:
# I think its fine now, right format, no NaNs, no duplicates
products_df.info()

<class 'pandas.DataFrame'>
Index: 10104 entries, 0 to 19325
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   sku       10104 non-null  str    
 1   name      10104 non-null  str    
 2   desc      10104 non-null  str    
 3   price     10104 non-null  float64
 4   in_stock  10104 non-null  int64  
 5   type      10104 non-null  str    
dtypes: float64(1), int64(1), str(4)
memory usage: 552.6 KB


In [54]:
# we can save a copy
products_df.to_csv('../data/cleaned_products_file.csv', index=False)